# Experiment — Generation of Videos from Text Prompts using a Pretrained Open-Source Video Generation Foundation Model

**Aim:** Use a pretrained, open-source video generation model to generate short videos directly from text prompts, without collecting a dataset or training the model, and preview and save the generated videos.

**Model:** `damo-vilab/text-to-video-ms-1.7b` via Hugging Face Diffusers.

**Important:** This notebook performs inference only. It does not train or fine-tune the model. Video generation is memory-intensive; select a GPU runtime in Google Colab. A T4 may work with memory-saving settings but is not guaranteed for every session. If the model fails to load or runs out of memory, restart the runtime and try a shorter prompt / fewer frames, or use a stronger GPU if available.

## 1. Select GPU runtime

In Google Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). Then run the cells from top to bottom.

In [ ]:
import torch, os, random, numpy as np
from IPython.display import display, Video, Markdown

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. This model is unlikely to run comfortably on CPU.")

## 2. Install required libraries

In [ ]:
!pip -q install -U diffusers transformers accelerate imageio imageio-ffmpeg safetensors

## 3. Load the pretrained text-to-video model

In [ ]:
import torch
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video

MODEL_ID = "damo-vilab/text-to-video-ms-1.7b"

# float16 + CPU offload reduce GPU memory use. First model download may take several minutes.
pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

print("Loaded pretrained model:", MODEL_ID)
print("Inference only: no training or fine-tuning is performed.")

## 4. Enter a text prompt and generate a short video

In [ ]:
# Edit this prompt to create your own video.
prompt = (
    "A cinematic wide shot of a small wooden boat floating on a calm lake at sunrise, "
    "soft golden light reflecting on the water, gentle mist, realistic nature documentary, "
    "smooth camera movement"
)

# Memory/time controls. Keep values small for a Colab demo.
num_frames = 16
num_inference_steps = 25
guidance_scale = 7.5
seed = 42

generator = torch.Generator(device="cpu").manual_seed(seed)

print("Prompt:", prompt)
print("Frames:", num_frames, "| Steps:", num_inference_steps, "| Seed:", seed)

with torch.inference_mode():
    result = pipe(
        prompt,
        num_frames=num_frames,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator
    )

frames = result.frames[0]
print("Generated frames:", len(frames))
print("Frame size:", frames[0].size)

## 5. Preview the generated video

In [ ]:
OUTPUT_PATH = "/content/generated_video.mp4"
export_to_video(frames, OUTPUT_PATH, fps=8)

display(Video(OUTPUT_PATH, embed=True, width=640))
print("Video saved to:", OUTPUT_PATH)

## 6. Download the generated MP4

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)

## 7. Generate another video (optional)

In [ ]:
# Change the prompt and rerun this cell to generate a second clip.
prompt_2 = (
    "A tiny robot exploring a lush futuristic greenhouse, glowing plants, sunlight through glass, "
    "slow tracking shot, colorful cinematic science fiction, detailed environment"
)
seed_2 = 123
generator_2 = torch.Generator(device="cpu").manual_seed(seed_2)

with torch.inference_mode():
    result_2 = pipe(
        prompt_2,
        num_frames=16,
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=generator_2
    )

frames_2 = result_2.frames[0]
OUTPUT_PATH_2 = "/content/generated_video_2.mp4"
export_to_video(frames_2, OUTPUT_PATH_2, fps=8)
display(Video(OUTPUT_PATH_2, embed=True, width=640))
print("Saved:", OUTPUT_PATH_2)

# Uncomment to download:
# files.download(OUTPUT_PATH_2)

## 8. Expected observations and conclusion

**Expected output:** a short MP4 clip generated from the supplied text prompt, displayed in the notebook and saved at `/content/generated_video.mp4`. The output will vary by prompt, random seed, model version, and runtime.

**Conclusion:** A pretrained open-source text-to-video foundation model can generate short video clips from natural-language prompts through inference, without collecting a dataset or training the model.

### Troubleshooting
- **CUDA out of memory:** Restart runtime; use fewer frames (e.g. 8), fewer inference steps (e.g. 15), and a shorter prompt. GPU memory availability varies.
- **Download/model access error:** Check Colab internet access and rerun the loading cell.
- **Slow generation:** Video diffusion is computationally expensive; wait for inference to finish.
- **No GPU:** Change runtime type to a GPU if one is available.
- **Model/package compatibility error:** Restart runtime after installation and run cells from the top. If a dependency release changes the pipeline API, the notebook may need a version adjustment.
- **Safety and rights:** Generate content you have the right to create; do not present synthetic video as authentic footage.